# Explainable AI— Opening the Black Box

**Module 15 · Lesson 15.2**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Black Box
- Feature Attribution
- LIME (Local Surrogates)
- SHAP (Shapley Values)
- Global vs Local
- Counterfactual Explanations

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## A decision with no reason is not an answer

> 💡 **Info**
>
> A model outputs a **score of 0.182** and the word **DENY**. That is the entire answer the applicant gets. A real model is millions of weights, and all of that computation collapses into one number that, by itself, explains nothing — the applicant cannot tell whether it was their income, their debt, their age, or a proxy for something they cannot change. For a low-stakes suggestion that is fine; for a loan, a medical triage, or a hiring screen it is a serious problem, because a decision you cannot explain is a decision you cannot *defend* — not to the person, not to a regulator, not to a court. GDPR Article 22 gives people the right not to be subject to purely automated high-stakes decisions without safeguards, and the EU AI Act (Articles 13 and 86) requires meaningful information about how a high-risk system reached its output. This lesson is the toolkit for producing that information. You will decompose an additive score into reasons, approximate a non-linear model in the neighborhood of one prediction, compute the fair Shapley split that handles feature interactions, separate what the model relies on overall from why *this* person got *this* answer, turn a reason into an action the applicant can take, and measure whether any of it is actually faithful to the model — because a plausible explanation that does not match the model is worse than none.

### 🎯 What you will be able to do after this lesson

- **Apply** feature attribution to decompose an additive model’s score into per-feature reasons.

- **Analyze** a single prediction locally with LIME, knowing a local weight is not a global truth.

- **Create** exact Shapley (SHAP) attributions and verify the efficiency property, and turn an attribution into a minimal counterfactual.

- **Evaluate** whether an explanation is faithful by measuring fidelity against the model, and reject a plausible-but-wrong one.

> 📦 **Info**
>
> ✅ Before you startThis lesson follows **15.1 (Bias Detection)**: there you learned that hiding race and gender does not stop discrimination because *proxy* features carry the signal — explainability is how you see *which* features the model actually used. It builds on **Module 03** (a model’s output is one number that hides millions of weights) and borrows the measurement discipline of **Module 14** (an explanation, like an eval, is only trustworthy if you measure how well it matches the model). Everything here is keyless arithmetic on one illustrative loan model.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🩺 **Analogy**
>
> An unexplained model decision is a **doctor who says “denied” and walks out**. Imagine being told your treatment is refused, or your test is positive, with no diagnosis, no reasoning, no idea what to do next — you cannot act on it, challenge it, or trust it. A good doctor names the finding, points to the evidence, and tells you what would change the outcome. Explainable AI is the toolkit that makes a model behave like the good doctor instead of the one who walks out: it turns a bare verdict into a diagnosis you can understand, verify, and act on. **Where it breaks down:** a doctor knows their own reasoning, while a model’s explanation is reconstructed after the fact — which is exactly why you must check that the reconstruction is *faithful* (Step 7).

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“The explanation is the true cause.”** An attribution shows what the *model* used, which can be a proxy correlated with the real cause — not the cause itself. Explainability reveals the model’s reasoning, not the world’s.
> - **“A plausible explanation is a correct one.”** Plausibility and faithfulness are different properties. A surrogate can tell a convincing story with a fidelity of almost zero — a confident alibi that does not match the model at all.

> 💡 **Info**
>
> ⚠️ Anti-patternThe explanation with no fidelity number, and the LLM chain-of-thought taken at face value. A team ships a dashboard of feature attributions that *look* authoritative but were never checked against the model, and treats a reasoning model’s stated “chain-of-thought” as its actual computation — when it is often a post-hoc rationalisation. Both give users and auditors false assurance: a story that reads well and reveals nothing. Every step below builds toward the discipline of measuring, not assuming, that an explanation is true.

---

## 🎯 Concept 1: The Black Box

### The Black Box

A model returns a decision and a score - not a reason. For a coffee recommendation nobody cares; for a loan, a diagnosis, or a parole decision, a verdict with no reason is not accountable, and under GDPR Art. 22 and the EU AI Act it is not lawful. Explainability recovers the “why” the raw output hides.

Start with the problem, stated plainly. A production model is an enormous function — millions of weights — and its output is a single number. That number is enough to make a decision but not to *justify* one. When the model denies a loan with a score of 0.182, the applicant, a regulator, and a lawyer all ask the same question — *why?* — and the score says nothing. This is the black-box problem, and it is not merely uncomfortable: for a high-risk decision it is a legal exposure. **GDPR Article 22** restricts solely-automated decisions with legal or similarly significant effects and requires safeguards; the **EU AI Act** (Articles 13 and 86) requires meaningful information about a high-risk system’s output and a right to explanation. So explainability is not a research curiosity — it is the mechanism that makes an automated decision defensible. The rest of the lesson is a toolkit for recovering the “why” the raw output throws away. The block runs the model, prints the opaque output, and states the gap, keyless.

> 🩺 **Analogy**
>
> The black box is **a verdict with no trial transcript**. A court that announced “guilty” and refused to say what evidence, which law, or what reasoning led there would not be a court — it would be an oracle, and no appeal would be possible. A model that outputs a score with no reason is that court. The whole point of a legal record is that the decision can be examined, challenged, and corrected; explainability is the transcript that makes a model’s decision reviewable in the same way. A number you cannot interrogate is a verdict you cannot appeal.

A loan model denies an applicant with a score of 0.182 and no other output. The applicant asks why. Is the score a sufficient answer?

**📝 `01_the_black_box.py`** - *runnable*

In [ ]:
# THE BLACK BOX: a model returns a DECISION and a score, but not a REASON. For a loan denial that is not just unsatisfying -
# under GDPR Art. 22 and the EU AI Act it can be illegal. Explainability turns one opaque number into an accountable story.
def loan_model(applicant):
    # a real model is millions of weights; the OUTPUT is a single score, and the score alone hides the "why"
    w = {"income": 0.9, "debt_ratio": -1.6, "credit_age": 0.4, "recent_defaults": -1.1}
    z = sum(w[f] * applicant[f] for f in w)      # standardized features, illustrative weights
    score = 1 / (1 + 2.71828 ** (-z))            # logistic squash to 0..1
    return round(score, 3), "APPROVE" if score >= 0.5 else "DENY"
applicant = {"income": 0.4, "debt_ratio": 0.8, "credit_age": 0.2, "recent_defaults": 0.6}
score, decision = loan_model(applicant)
print("model output:  score = {}   decision = {}".format(score, decision))
print("the applicant asks: WHY was I denied? The score {} answers nothing.".format(score))
print("A decision without a reason is not accountable - and for a high-risk system it is not lawful. That is the gap XAI fills.")

# Output:
# model output:  score = 0.182   decision = DENY
# the applicant asks: WHY was I denied? The score 0.182 answers nothing.
# A decision without a reason is not accountable - and for a high-risk system it is not lawful. That is the gap XAI fills.

- The model emits a **score** (0.182) and a **decision** (DENY) — and nothing about why.
- The score is a single number collapsing millions of weights; by itself it explains nothing.
- The applicant, a regulator, and a lawyer all ask “why” — and the raw output has no answer.
- A decision without a reason is not accountable and, for a high-risk system, not lawful (GDPR Art. 22, EU AI Act) — the gap XAI fills.

#### 💡 What just happened

⚡ What just happened? A model returns a decision and a score but not a reason; for a high-risk decision a verdict with no reason is not accountable and can be unlawful (GDPR Art. 22, EU AI Act). Tradeoff: none - recovering the reason is required. Next: the simplest way to recover it, when the model allows it.

- A model emits a score and a DENY decision
- A big unanswered ‘why?’ floats over the opaque output

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Feature Attribution

### Feature Attribution

When the model is ADDITIVE, the reason decomposes exactly: each feature’s contribution is its weight times its value, and the contributions sum to the score. It is the simplest and most faithful explanation there is - no approximation, no surrogate - and it works whenever the model is linear in its features.

The easiest explanation to trust is the one that is exactly true, and for an **additive** model it exists. If the score is a weighted sum of features, then each feature’s **contribution** is simply its weight times its value, and the contributions add up to the score with nothing left over. There is no surrogate, no perturbation, no approximation error — the decomposition is the model. For the denied applicant the contributions are debt_ratio at -1.28, recent_defaults at -0.66, income at +0.36, and credit_age at +0.08, and they sum to a log-odds of -1.5, which is what pushed the score below the threshold. Now the denial has a reason stated in the applicant’s own features: *high debt ratio and recent defaults outweighed income*. This is the gold standard when the model permits it — and it is exactly why interpretable “glass-box” models (linear models, generalised additive models) are often preferred for high-stakes decisions: their explanation is free and faithful by construction. The catch, which the next step confronts, is that most powerful models are *not* additive. The block computes and ranks the contributions, keyless.

> 🧾 **Analogy**
>
> Feature attribution is an **itemized receipt for the decision**. Instead of a total that says only “forty-seven dollars,” the receipt lists every line — the entree, the wine, the service charge — each with its own signed amount, and the lines add up to exactly the total. An additive model’s attribution is that receipt: debt_ratio, defaults, income, and credit each show their signed contribution, and they sum to the score. You can see at a glance which line was the big one, and because it is a receipt and not a guess, the numbers reconcile perfectly. The limitation is the same as a receipt’s: it only works when the total really is the sum of the parts.

For an additive model, the contributions are debt_ratio -1.28, recent_defaults -0.66, income +0.36, credit_age +0.08, summing to -1.5. What is the reason for the denial?

**📝 `02_feature_attribution.py`** - *runnable*

In [ ]:
# FEATURE ATTRIBUTION: for an ADDITIVE model the reason decomposes exactly - each feature's contribution is its weight times
# its value, and the contributions sum to the score. This is the simplest, most faithful explanation when the model allows it.
w = {"income": 0.9, "debt_ratio": -1.6, "credit_age": 0.4, "recent_defaults": -1.1}
applicant = {"income": 0.4, "debt_ratio": 0.8, "credit_age": 0.2, "recent_defaults": 0.6}
contrib = {f: round(w[f] * applicant[f], 3) for f in w}
z = round(sum(contrib.values()), 3)
print("per-feature contribution to the log-odds (weight x value):")
for f, c in sorted(contrib.items(), key=lambda kv: kv[1]):
    bar = ("-" if c < 0 else "+") * max(1, int(abs(c) * 20))
    print("  {:<16} {:+.3f}  {}".format(f, c, bar))
print()
print("total log-odds z = {}  ->  the two NEGATIVE drivers are debt_ratio ({}) and recent_defaults ({}).".format(
      z, contrib["debt_ratio"], contrib["recent_defaults"]))
print("Now the denial has a reason: high debt ratio and recent defaults outweighed income. That is an explanation you can act on.")

# Output:
# per-feature contribution to the log-odds (weight x value):
#   debt_ratio       -1.280  -------------------------
#   recent_defaults  -0.660  -------------
#   credit_age       +0.080  +
#   income           +0.360  +++++++
#
# total log-odds z = -1.5  ->  the two NEGATIVE drivers are debt_ratio (-1.28) and recent_defaults (-0.66).
# Now the denial has a reason: high debt ratio and recent defaults outweighed income. That is an explanation you can act on.

- Each feature’s **contribution** is its weight times its value; the contributions sum to the score exactly.
- The two negative drivers — debt_ratio (-1.28) and recent_defaults (-0.66) — outweighed income (+0.36).
- The denial now has a reason in the applicant’s own features: high debt and recent defaults.
- This is the most faithful explanation there is — but only when the model is additive; most powerful models are not.

#### 💡 What just happened

⚡ What just happened? For an additive model the reason decomposes exactly: contribution = weight x value, and the contributions sum to the score. Tradeoff: it only works when the model is linear in its features - which is why glass-box models are favoured for high-stakes use. Next: what to do when the model is not additive.

- Signed per-feature contribution bars
- They sum exactly to the score (the log-odds)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: LIME (Local Surrogates)

### LIME (Local Surrogates)

Real models are non-linear, so you approximate the model LOCALLY: perturb one prediction, see how the output moves, and fit a simple linear model in that neighborhood. The local sensitivities explain THAT instance - not the model globally. On a non-linear model the same feature has different local weights at different points.

Most models worth using are non-linear — features interact, effects saturate, and there is no single global weight for “income.” So instead of explaining the whole model at once, **LIME** (Local Interpretable Model-agnostic Explanations) explains *one prediction*: it perturbs the input around that instance, watches how the model’s output moves, and fits a simple linear model in that small neighborhood. The weights of that local model are the explanation for *this* applicant. In the worked example, near the denied applicant income has a local weight of -0.1 — but at a different applicant with low debt, income’s local weight is +0.5. Same feature, opposite local effect, because the model is non-linear and the two applicants sit on different parts of its surface. That is the essential discipline: a LIME weight is a statement about a *point*, not the model. Reading a local weight as a global truth is the single most common way practitioners misuse it. The block probes the neighborhood by finite differences (the local-slope idea behind LIME) and shows the weight changing between two points, keyless.

> 📏 **Analogy**
>
> LIME is a **tangent line touching a curve**. A curved road has no single slope, but at any one point you can lay a straight ruler against it and read the slope *right there* — and that tangent tells you the direction accurately for a few steps in either direction, then drifts wrong. Move to a different point on the curve and the tangent tilts a different way. LIME fits that local straight-line ruler to the model’s curved surface at one prediction; the local weights are the tangent’s slope. Trusting one tangent to describe the entire curve is exactly the mistake of reading a local explanation as a global one.

A LIME explanation for one applicant gives income a local weight of -0.1. Can you conclude income hurts every applicant?

**📝 `03_lime_local.py`** - *runnable*

In [ ]:
# LIME: real models are NOT additive, so you approximate the model LOCALLY. Perturb one prediction, see how the output moves,
# and fit a simple linear model in that neighborhood. The local sensitivities are the explanation FOR THIS INSTANCE (not globally).
def model(x):  # a NON-linear model: income and debt interact (their product matters), so no single global weight exists
    return round(0.6 * x["income"] - 1.5 * x["debt_ratio"] - 0.9 * x["income"] * x["debt_ratio"], 3)
base = {"income": 0.4, "debt_ratio": 0.8}
f0 = model(base)
eps = 0.01
local = {}
for feat in base:
    up = dict(base); up[feat] += eps          # probe the neighborhood: nudge one feature, measure the output change
    local[feat] = round((model(up) - f0) / eps, 3)   # local slope = LIME-style local weight
print("prediction at this point: {}".format(f0))
print("local sensitivity (how the output moves as each feature nudges up, near THIS applicant):")
for feat, slope in local.items():
    print("  {:<12} {:+.3f}".format(feat, slope))
far = {"income": 0.9, "debt_ratio": 0.1}
far_slope = round((model({"income": 0.91, "debt_ratio": 0.1}) - model(far)) / eps, 3)
print()
print("income's local weight here is {} but at a different applicant (low debt) it is {} - the SAME feature, different local".format(local["income"], far_slope))
print("effect, because the model is non-linear. LIME explains ONE prediction; do not read a local weight as a global truth.")

# Output:
# prediction at this point: -1.248
# local sensitivity (how the output moves as each feature nudges up, near THIS applicant):
#   income       -0.100
#   debt_ratio   -1.900
#
# income's local weight here is -0.1 but at a different applicant (low debt) it is 0.5 - the SAME feature, different local
# effect, because the model is non-linear. LIME explains ONE prediction; do not read a local weight as a global truth.

- LIME approximates the model **locally**: nudge each feature near one prediction and read the resulting output change.
- The **local slope** is the explanation for THAT instance — here income’s local weight is -0.1.
- At a different applicant (low debt), income’s local weight is +0.5 — the same feature, a different local effect.
- The model is non-linear, so a local weight is **not** a global truth; do not generalise a LIME explanation.

#### 💡 What just happened

⚡ What just happened? LIME explains one prediction by fitting a simple linear model in its neighborhood; the local weights describe that point, and on a non-linear model the same feature has different local weights elsewhere. Tradeoff: a local explanation cannot be read globally. Next: the attribution that is provably fair, even under feature interactions.

- A tangent slope touching a non-linear model curve at one point
- A probe slider moves the point and the local slope changes

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: SHAP (Shapley Values)

### SHAP (Shapley Values)

SHAP is the game-theoretic answer. A feature’s SHAP value is its average marginal contribution over EVERY order of adding features - the unique attribution that is fair (Shapley 1953, Lundberg-Lee 2017). It handles interactions correctly, and the values sum exactly to prediction minus baseline (the efficiency property).

Feature attribution is exact but only for additive models; LIME is general but only local and approximate. **SHAP** gives a general attribution with a fairness guarantee, by borrowing from cooperative game theory. Treat the features as players in a game whose payoff is the prediction. A feature’s **Shapley value** is its average *marginal contribution* across every possible order in which features could be added to the model — the amount it adds, on average, when it joins. Shapley (1953) proved this is the *unique* attribution satisfying a set of fairness axioms (efficiency, symmetry, dummy, additivity), and Lundberg and Lee (2017) brought it to machine learning as SHAP. Two properties make it the default for tabular models. First, it handles **interactions** correctly — when income and debt matter *together*, SHAP splits that synergy fairly between them, which naive weight-times-value cannot. Second, **efficiency**: the SHAP values sum exactly to the prediction minus a **baseline** — the model’s average output when no feature is credited (the empty coalition, here 0.10). In the worked example the values are income 0.317, debt_ratio 0.267, credit_age 0.117, and they sum to 0.70, which is exactly prediction 0.80 minus baseline 0.10. The block computes them exactly over all coalitions with `itertools` — no `shap` library, no approximation — and verifies the efficiency sum, keyless.

> 🍽️ **Analogy**
>
> SHAP is **splitting a restaurant bill fairly among diners who shared dishes**. Some ordered alone, some split a platter, some only picked at the shared sides — and a fair split cannot just divide by heads or by what each ordered solo, because the shared dishes create value that belongs to several people at once. The Shapley value is the mathematically fair answer: for each diner, average what they add to the bill across every order in which people could have arrived. It splits the shared platter between the people who actually shared it. SHAP does exactly this with feature interactions — the “shared dish” of income-and-debt-together is divided fairly between them, and everyone’s share sums to the exact total.

Your model has feature interactions. Why prefer SHAP over naive weight-times-value attribution?

**📝 `04_shap_shapley.py`** - *runnable*

In [ ]:
# SHAP: the game-theoretic answer. A feature's SHAP value is its average marginal contribution over EVERY order of adding
# features - the unique attribution that is fair (Shapley 1953). Exact here for 3 features via all coalitions; it needs no scipy.
from itertools import combinations
from math import factorial
FEATURES = ["income", "debt_ratio", "credit_age"]
v = {(): 0.10, ("income",): 0.30, ("debt_ratio",): 0.25, ("credit_age",): 0.15,
     ("income", "debt_ratio"): 0.60, ("income", "credit_age"): 0.40, ("debt_ratio", "credit_age"): 0.35,
     ("income", "debt_ratio", "credit_age"): 0.80}   # v(S) = model output with only S present (rest at baseline); income+debt synergise
def val(S): return v[tuple(f for f in FEATURES if f in S)]
n = len(FEATURES)
shap = {}
for f in FEATURES:
    others = [x for x in FEATURES if x != f]
    phi = 0.0
    for k in range(len(others) + 1):
        for S in combinations(others, k):
            weight = factorial(len(S)) * factorial(n - len(S) - 1) / factorial(n)
            phi += weight * (val(set(S) | {f}) - val(set(S)))   # marginal contribution of f given coalition S
    shap[f] = round(phi, 3)
print("SHAP value per feature (average marginal contribution over all orderings):")
for f in FEATURES:
    print("  {:<12} {:+.3f}".format(f, shap[f]))
total, base, pred = round(sum(shap.values()), 3), v[()], v[("income", "debt_ratio", "credit_age")]
print()
print("sum of SHAP values = {}  ==  prediction {} - baseline {} = {}   (the EFFICIENCY property holds exactly).".format(
      total, pred, base, round(pred - base, 3)))
print("SHAP handles the income+debt interaction fairly - it splits the synergy between them, which naive weight x value cannot.")

# Output:
# SHAP value per feature (average marginal contribution over all orderings):
#   income       +0.317
#   debt_ratio   +0.267
#   credit_age   +0.117
#
# sum of SHAP values = 0.701  ==  prediction 0.8 - baseline 0.1 = 0.7   (the EFFICIENCY property holds exactly).
# SHAP handles the income+debt interaction fairly - it splits the synergy between them, which naive weight x value cannot.

- A feature’s **SHAP value** is its average marginal contribution over every order of adding features (computed exactly over all coalitions).
- The values are income 0.317, debt_ratio 0.267, credit_age 0.117 — the fair split of the prediction.
- **Efficiency**: they sum to 0.70, exactly prediction 0.80 minus baseline 0.10.
- SHAP splits the income-and-debt interaction fairly between them — something naive weight-times-value cannot do.

#### 💡 What just happened

⚡ What just happened? A SHAP value is a feature’s average marginal contribution over all orderings - the unique fair attribution (Shapley 1953) - and the values sum exactly to prediction minus baseline. Tradeoff: exact SHAP is exponential in features, so real tools approximate it (KernelSHAP, TreeSHAP). Next: one SHAP value explains one prediction, so how do you get overall importance?

- Feature coalitions build up marginal contributions
- The SHAP values sum to prediction minus baseline (efficiency)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Global vs Local

### Global vs Local

A SHAP value explains ONE prediction (local). Average the ABSOLUTE SHAP values across many predictions and you get GLOBAL feature importance - what the model relies on overall. The global ranking can differ from any single local one, so you report both: global for the model, local for the individual.

Everything so far explains a single prediction — but you also need to know what the model relies on *overall*, for documentation, debugging, and governance. The bridge is simple: a SHAP value is **local** (one instance), and if you take the **mean absolute SHAP value** of each feature across many predictions, you get **global** feature importance — how much that feature moves the model’s output on average. In the worked example the global ranking puts debt_ratio first (mean absolute SHAP 0.30), then income (0.217), then credit_age (0.20). But global importance is an average, and averages hide individuals: for applicant_2 the biggest local driver is *credit_age* at +0.42, even though credit_age is the least important feature globally. There is no contradiction — global tells you what the model leans on across everyone, local tells you why *this* person got *this* decision, and a responsible explanation reports both. Confusing the two — quoting a global importance to explain an individual decision, or generalising one local explanation into a claim about the model — is a common and consequential mistake. The block aggregates SHAP into global importance and surfaces the local-vs-global disagreement, keyless.

> 🌲 **Analogy**
>
> Global versus local is **the forest and one tree**. Fly over a forest and you can say, truthfully, “this is mostly oak” — that is global importance, the average character of the whole. But stand next to one particular tree and it might be a birch; describing *that* tree as an oak because the forest is mostly oak would be wrong. Global importance is the view from the plane; a local explanation is standing at one tree. Both are true, and each answers a different question — “what is this forest made of” versus “what is this tree” — so you never substitute one for the other.

Global importance ranks debt_ratio first, but applicant_2’s decision was driven by credit_age. Is that a contradiction?

**📝 `05_global_vs_local.py`** - *runnable*

In [ ]:
# GLOBAL vs LOCAL: a SHAP value explains ONE prediction (local). Average the ABSOLUTE SHAP values across many predictions and
# you get GLOBAL feature importance - which feature matters most overall. The global ranking can differ from any single local one.
instances = {   # per-instance SHAP values for three applicants (from the local explainer, illustrative)
    "applicant_1": {"income": 0.32, "debt_ratio": -0.40, "credit_age": 0.10},
    "applicant_2": {"income": 0.05, "debt_ratio": -0.15, "credit_age": 0.42},   # the outlier: credit_age drove this one
    "applicant_3": {"income": 0.28, "debt_ratio": -0.35, "credit_age": 0.08}}
features = ["income", "debt_ratio", "credit_age"]
global_imp = {f: round(sum(abs(inst[f]) for inst in instances.values()) / len(instances), 3) for f in features}
print("GLOBAL importance (mean |SHAP| across applicants):")
for f, imp in sorted(global_imp.items(), key=lambda kv: -kv[1]):
    print("  {:<12} {:.3f}".format(f, imp))
print()
top_local = max(instances["applicant_2"], key=lambda f: abs(instances["applicant_2"][f]))
top_global = max(global_imp, key=global_imp.get)
print("Globally the biggest driver is {}. But for applicant_2 the biggest local driver is {} (SHAP {:+.2f}).".format(
      top_global, top_local, instances["applicant_2"][top_local]))
print("Global tells you what the model relies on overall; local tells you why THIS person got THIS decision. You need both.")

# Output:
# GLOBAL importance (mean |SHAP| across applicants):
#   debt_ratio   0.300
#   income       0.217
#   credit_age   0.200
#
# Globally the biggest driver is debt_ratio. But for applicant_2 the biggest local driver is credit_age (SHAP +0.42).
# Global tells you what the model relies on overall; local tells you why THIS person got THIS decision. You need both.

- **Global importance** is the mean absolute SHAP value per feature across many predictions — debt_ratio (0.30) leads.
- But the average hides individuals: for applicant_2 the biggest **local** driver is credit_age (+0.42).
- There is no contradiction — global is what the model leans on overall; local is why this person got this decision.
- Report **both**: global for the model’s behaviour, local for an individual’s explanation.

#### 💡 What just happened

⚡ What just happened? A SHAP value is local; mean absolute SHAP across instances gives global importance, and the global ranking can differ from any single local one. Tradeoff: you must report both and not substitute one for the other. Next: turning a local reason into something the applicant can actually act on.

- Mean-absolute-SHAP bars for global importance
- One applicant’s local bars rank features differently

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Counterfactual Explanations

### Counterfactual Explanations

The most actionable explanation: what is the SMALLEST change that would flip the decision? A counterfactual answers the applicant’s real question - “what do I need to do?” - and regulators favour it because it is concrete and checkable, unlike a raw attribution which only says what drove the outcome.

An attribution tells you *why* the model decided, but the applicant usually wants to know something more useful: *what would I have to change to get a different decision?* That is a **counterfactual explanation** (Wachter, Mittelstadt, and Russell, 2017): the smallest change to the input that flips the outcome. For the denied applicant, the log-odds is -1.5 and it needs to reach 0 to cross into APPROVE, a gap of 1.5. The block computes, for each feature, the change that would close that gap: increase income by 1.67 standard units, or reduce debt_ratio by 0.94, or reduce recent_defaults by 1.36. The **smallest actionable** one wins — reducing debt_ratio by 0.94 — and that is the explanation the applicant can actually act on. Counterfactuals are increasingly the form regulators prefer, for two reasons: they are **concrete** (a specific change, not a diffuse ranking) and they are **checkable** (anyone can verify that the change flips the decision). A good counterfactual also respects *actionability* — suggesting “reduce your debt” is useful, “be born earlier” is not — so real systems constrain counterfactuals to features the person can change. The block finds the minimal single-feature counterfactual, keyless.

> 🚩 **Analogy**
>
> A counterfactual is the **“you needed twelve more points to pass”** on a failed exam. Being told your reasoning was weak on section three (the attribution) is informative; being told exactly how many more marks, and where, you needed to pass (the counterfactual) is *actionable* — it tells you precisely what to do differently next time. The applicant denied a loan does not primarily want a diagnosis of their finances; they want the passing mark: the smallest concrete change that would have flipped the decision. And like an exam boundary, it is checkable — anyone can confirm that twelve more points crosses the line.

A denied applicant asks what they need to change to get approved. Which explanation answers them?

**📝 `06_counterfactual.py`** - *runnable*

In [ ]:
# COUNTERFACTUAL EXPLANATIONS: the most actionable form - "what is the SMALLEST change that would flip the decision?" It answers
# the applicant's real question ("what do I need to do?") and is the form regulators favour, because it is concrete and checkable.
w = {"income": 0.9, "debt_ratio": -1.6, "credit_age": 0.4, "recent_defaults": -1.1}
applicant = {"income": 0.4, "debt_ratio": 0.8, "credit_age": 0.2, "recent_defaults": 0.6}
z = sum(w[f] * applicant[f] for f in w)          # current log-odds (score below 0.5 means DENY: z < 0)
print("current log-odds z = {:.3f}  ->  DENY (needs z >= 0 to flip to APPROVE).".format(z))
gap = 0 - z                                       # how much log-odds must increase to cross the threshold
print("to flip the decision, the log-odds must rise by {:.3f}. Cheapest single-feature counterfactuals:".format(gap))
for f in ["income", "debt_ratio", "recent_defaults"]:
    delta = gap / w[f]                            # change in this feature needed to close the gap (respect the sign)
    direction = "increase" if delta > 0 else "reduce"
    print("  {} {:<16} by {:.2f}  (standardized)".format(direction, f, abs(delta)))
print()
print("The smallest actionable change wins: reducing debt_ratio by {:.2f} flips APPROVE. A counterfactual is an explanation the".format(abs(gap / w["debt_ratio"])))
print("applicant can act on and a regulator can verify - unlike a raw attribution, it says exactly what would have changed the outcome.")

# Output:
# current log-odds z = -1.500  ->  DENY (needs z >= 0 to flip to APPROVE).
# to flip the decision, the log-odds must rise by 1.500. Cheapest single-feature counterfactuals:
#   increase income           by 1.67  (standardized)
#   reduce debt_ratio       by 0.94  (standardized)
#   reduce recent_defaults  by 1.36  (standardized)
#
# The smallest actionable change wins: reducing debt_ratio by 0.94 flips APPROVE. A counterfactual is an explanation the
# applicant can act on and a regulator can verify - unlike a raw attribution, it says exactly what would have changed the outcome.

- The applicant’s log-odds is -1.5 and must reach 0 to flip to APPROVE — a gap of 1.5 to close.
- For each feature, the change that closes the gap: income +1.67, debt_ratio -0.94, recent_defaults -1.36.
- The **smallest actionable** change wins: reducing debt_ratio by 0.94 flips the decision.
- A counterfactual is concrete and checkable — the form regulators favour, and the one the applicant can act on.

#### 💡 What just happened

⚡ What just happened? A counterfactual is the smallest change that flips the decision - the actionable, checkable explanation regulators favour, and the one an applicant actually needs. Tradeoff: you must constrain it to features the person can change (actionability). Next: none of this matters if the explanation is not actually faithful to the model.

- A feature slider adjusts one input
- Reducing debt_ratio flips DENY to APPROVE at the boundary

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Faithfulness

### Faithfulness

An explanation can be plausible and still WRONG. FIDELITY - how well the explanation tracks the real model on held-out perturbations - separates a real reason from a confident alibi. Always report fidelity; a low-fidelity explanation (and, in LLMs, an unfaithful chain-of-thought) is worse than none because it manufactures false confidence.

Here is the step that separates explainability from theater. Every method so far — attribution, LIME, SHAP, counterfactuals — produces an explanation, but an explanation is only trustworthy if it actually *matches the model*. A surrogate can be perfectly plausible and completely wrong: it tells a clean story that a human finds convincing while bearing no relationship to what the model computed. The defense is to measure **fidelity** — how well the explanation predicts the model’s behaviour on held-out perturbations, reported as an R-squared. In the worked example a faithful surrogate scores 0.988 (it tracks the model closely, trust it) while an unfaithful one scores 0.025 (nearly flat, ignoring the real drivers — a plausible alibi, reject it). The same failure appears in LLMs as an **unfaithful chain-of-thought**: a reasoning model’s stated steps often rationalise an answer rather than reveal the computation that produced it, and “attention is not explanation” is the same lesson generalised. The rule is simple and non-negotiable: *always report fidelity with an explanation.* A low-fidelity explanation is worse than none, because none leaves you appropriately uncertain while a confident-but-false one manufactures trust you should not have. For a high-risk decision under the EU AI Act (Articles 13 and 86), fidelity is the difference between a meaningful explanation and a liability. The block computes fidelity for a faithful and an unfaithful surrogate, keyless.

> 🥷 **Analogy**
>
> Faithfulness is the difference between an **alibi and the truth**. A suspect can give a perfectly coherent, detailed, plausible account of their evening — and it can be entirely fabricated. What makes an account trustworthy is not how well it hangs together but whether it matches the independent evidence: the receipts, the camera footage, the timestamps. An explanation is the account; fidelity is the check against the evidence — does this story actually predict what the model did on cases you held back? A high-fidelity explanation is corroborated by the footage; a low-fidelity one is a smooth alibi that falls apart the moment you check it. Never accept the alibi without checking the tape.

A surrogate explanation looks plausible but its fidelity against the model is an R-squared of 0.025. Should you trust it?

**📝 `07_faithfulness.py`** - *runnable*

In [ ]:
# FAITHFULNESS: an explanation can be PLAUSIBLE and still WRONG. A surrogate that does not track the real model is a confident
# alibi, not a reason. Measure FIDELITY (how well the explanation predicts the model on held-out perturbations) before you trust it.
model_outputs   = [0.81, 0.62, 0.44, 0.90, 0.55, 0.33, 0.70, 0.48]   # the real model on 8 perturbations
good_surrogate  = [0.79, 0.64, 0.42, 0.88, 0.57, 0.35, 0.68, 0.50]   # a faithful local linear surrogate
bad_surrogate   = [0.60, 0.58, 0.61, 0.59, 0.62, 0.57, 0.60, 0.58]   # plausible-looking but flat: ignores the real drivers
def r2(y, yhat):
    mean = sum(y) / len(y)
    ss_res = sum((a - b) ** 2 for a, b in zip(y, yhat))
    ss_tot = sum((a - mean) ** 2 for a in y)
    return round(1 - ss_res / ss_tot, 3)
print("fidelity (R^2 of the surrogate against the real model on held-out perturbations):")
print("  faithful surrogate:  R^2 = {}   -> trust its explanation".format(r2(model_outputs, good_surrogate)))
print("  unfaithful surrogate: R^2 = {}  -> reject it: plausible story, wrong reason".format(r2(model_outputs, bad_surrogate)))
print()
print("Always report fidelity with an explanation. A low-fidelity explanation (and, in LLMs, an unfaithful chain-of-thought that")
print("rationalises rather than reveals) is worse than none - it manufactures false confidence. The EU AI Act (Art. 13, 86) requires")
print("meaningful explanations for high-risk decisions, so fidelity is not optional - it is the difference between XAI and theater.")

# Output:
# fidelity (R^2 of the surrogate against the real model on held-out perturbations):
#   faithful surrogate:  R^2 = 0.988   -> trust its explanation
#   unfaithful surrogate: R^2 = 0.025  -> reject it: plausible story, wrong reason
#
# Always report fidelity with an explanation. A low-fidelity explanation (and, in LLMs, an unfaithful chain-of-thought that
# rationalises rather than reveals) is worse than none - it manufactures false confidence. The EU AI Act (Art. 13, 86) requires
# meaningful explanations for high-risk decisions, so fidelity is not optional - it is the difference between XAI and theater.

- **Fidelity** is the R-squared of the surrogate’s predictions against the real model on held-out perturbations.
- The faithful surrogate scores 0.988 — it tracks the model, so trust its explanation.
- The unfaithful surrogate scores 0.025 — nearly flat, a plausible story that ignores the real drivers; reject it.
- Always report fidelity; a low-fidelity explanation (and an unfaithful LLM chain-of-thought) is worse than none — the EU AI Act (Art. 13, 86) requires meaningful, faithful explanations.

#### 💡 What just happened

⚡ What just happened? An explanation is only trustworthy if it is faithful; fidelity (how well it tracks the model on held-out perturbations) separates a real reason from a confident alibi - 0.988 trust, 0.025 reject. Tradeoff: measuring fidelity is extra work, but a low-fidelity explanation is worse than none. That is the whole toolkit: decompose, approximate locally, split fairly, separate global from local, give a counterfactual, and measure fidelity.

> 📦 **Info**
>
> 🚫 Fidelity is necessary, not sufficient: explanations can be gamedMeasuring fidelity catches the explanation you got *wrong by accident*; it does not, on its own, catch one built to *deceive*. Post-hoc explainers can be **fairwashed**: a biased model is deliberately wrapped so that the LIME or SHAP explanation looks innocuous — attributing the decision to neutral features while the model quietly relies on a protected proxy (the 15.1 problem). Because LIME and SHAP explain a model an auditor cannot see, an adversary who controls the model can engineer a plausible, even high-fidelity-looking, explanation that hides the real driver. Post-hoc explanations are also *unstable* — small input changes can swing them. The defenses are structural: prefer intrinsically interpretable (glass-box) models for high-stakes use, demand explanations from an independent party, and pair them with the bias tests from Lesson 15.1 — an explanation is a claim to be audited, not a certificate of good behaviour.

- A surrogate-vs-model scatter with the diagonal
- Points hug the diagonal (faithful) or scatter off it (unfaithful)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture — opening the black box
> Recovering the “why” is a toolkit, chosen by the model and the question. **Decompose** when the model is additive — feature attribution is exact and free (Step 2). **Approximate locally** when it is not — LIME explains one prediction, and a local weight is not a global truth (Step 3). **Split fairly** under interactions — SHAP’s Shapley values are the unique fair attribution and sum to prediction minus baseline (Step 4). **Separate global from local** — mean absolute SHAP for the model, a single local explanation for the individual (Step 5). **Give a counterfactual** when the person needs an action — the smallest change that flips the decision (Step 6). And above all, **measure fidelity** — a plausible explanation that does not track the model is a liability, not a reason (Step 7). Ask two questions of any explanation you ship: does it answer the question the person actually asked, and have you measured that it is faithful to the model?

**📦 **The XAI-method picker****

| Method | What it answers | When to use it | The caveat |
|---|---|---|---|
| Feature attribution | Which features drove this additive score? | Linear / glass-box models | Only exact for additive models |
| LIME | What does the model do near this one prediction? | Any model, one instance, quick | Local only - do not read as global |
| SHAP | What is each feature’s fair share of the prediction? | Tabular models, interactions, the default | Exact is exponential; tools approximate it |
| Counterfactual | What is the smallest change that flips the decision? | Giving a person an actionable answer | Must be constrained to changeable features |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now recover and, crucially, validate the reason behind a model’s decision. This sits between the two lessons around it: whether the model treats groups equally was bias detection, back in Lesson 15.1, and the law that *requires* these explanations for high-risk systems is covered in Lesson 15.6 (the regulatory landscape). What the model learned about the specific people in its training data we come back to in Lesson 15.4 (privacy); the program that mandates explainability as one accountable process closes the module in Lesson 15.7 (governance). Looking inside the network itself — sparse features and circuits, or mechanistic interpretability — is the frontier beyond these post-hoc methods. The reference tools are [SHAP](https://github.com/shap/shap) and [LIME](https://github.com/marcotcr/lime), and the foundational results are the [SHAP paper](https://arxiv.org/abs/1705.07874) and [Wachter et al. on counterfactuals](https://arxiv.org/abs/1711.00399).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Explainable AI— Opening the Black Box**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-15_2.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-15.2.html` - regenerate this notebook whenever the source HTML is updated.*
